# Cleveland Department of Health Complaints Analysis
An analysis of reports made to the Cleveland department of health based on data pulled from the OpenData CLE portal

## Setup

In [46]:
import os

import pandas as pd
from tabulate import tabulate
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [47]:
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg://{os.environ['PG_USER']}:{os.environ['PG_PASSWORD']}"
    f"@localhost:5432/{os.environ['PG_DB']}"
)

## Data Quality Log

| Column/Issue | What I found | How I handled it | Details |
| ------------ | ------------ | ---------------- | ------- |
| `mac_complaint_id` | Mostly null, mixed formats (date-seq, truncated, six-digit) | Excluded from analysis | Too sparse to support any conclusion and not needed for this analysis |
| `complaint_number` | Multiple rows sharing same complaint number differing only in dw_parcel | Deduplicated to one row per complaint | `dw_parcel` is a weekly-refreshed geocoding artifact not needed for this analysis |
| `complaint_type` | 21 rows without complaint type that were mostly empty | Excluded from analysis | Rows did not provide enough data to draw any conclusions |
| `complaint_status` | One row with `NULL` status | Excluded via `IS NOT NULL` operator | [see Question 1](#question-1-most-frequent-complaint-types) |
| No `resolved_datetime` column | No timestamp data for when complaints are resolved | only did duration calculations on currently open complaints | [see Question 2](#question-2-currently-open-complaints-with-longest-average-open-time-by-type-5-complaints-per-type-threshold) |
| `submit_datetime` | Portal docs give no timezone, format only | Parsed and stored as simple `timestamp` | [see Question 2](#question-2-currently-open-complaints-with-longest-average-open-time-by-type-5-complaints-per-type-threshold) |


## Question 1: Top 10 most frequent complaint types

**Data quality note:** `complaint_status` contains one row with a `NULL` value. Utilizing `IS NOT NULL` makes sure to only return rows that have a valid complaint status attached.

In [48]:
query = """
SELECT complaint_type, COUNT(complaint_type) as number_of_complaints
FROM complaints_raw
WHERE complaint_type IS NOT NULL
GROUP BY complaint_type
ORDER BY number_of_complaints DESC
LIMIT 10;
"""

df = pd.read_sql_query(query, con=engine)

display(df)

,complaint_type,number_of_complaints
0,Accumulation of Garbage,14244
1,High Grass/Weeds,7494
2,Rodent Abatement,3958
3,Food Complaints,3473
4,Mold,2564
5,Insect/Vermin Infestation,2218
6,Animal Nuisance,1370
7,Other,1270
8,Unsanitary Conditions,1176
9,Health Condition,736


**Findings:** The number one complaint type made to the city was for excessive accumulations of garbage, nearly double the number of complaints of the second most common complaint of high grass/weeds.

## Question 2: Currently open complaints with longest average open time by type (5+ complaints per type threshold)

**Data quality notes:** 
- `submit_datetime` has no documented timezone (city portal
only specifies the format, not the zone). Values appear consistent with EST
daylight hours but this is guessed, not confirmed. Stored as a `timestamp`
(no offset) rather than asserting a timezone the data doesn't explicitly mention.
- No timestamp data exists for when complaints are resolved, so duration of time that complaints are open for after being resolved can not be determined. Average open durations are therefore calculated only for complaints that are currently open.
- Complaints with statuses of 'Inspected' and 'Reactivate' were excluded as these seem to correlate with addressed complaints. Statuses 'Invalid Info', 'Outside City Limits', and 'Resolved' are also outside the scope of this query.

In [49]:
query = """
WITH active_complaints AS (
    SELECT complaint_type, (LOCALTIMESTAMP - submit_datetime) AS duration_open FROM complaints
    WHERE complaint_status IN ('Active', 'New')
)

SELECT complaint_type,
    AVG(duration_open) AS avg_length_open,
    MAX(duration_open) AS longest_open,
    TRUNC(EXTRACT(EPOCH FROM MAX(duration_open)) / EXTRACT(EPOCH FROM AVG(duration_open)), 2) AS max_to_avg_ratio,
    COUNT(complaint_type) AS sample_size
FROM active_complaints
GROUP BY complaint_type
HAVING COUNT(complaint_type) >= 5
ORDER BY avg_length_open DESC;
"""

df = pd.read_sql_query(query, con=engine)

display(df)

,complaint_type,avg_length_open,longest_open,max_to_avg_ratio,sample_size
0,Food Complaints,688 days 10:30:02.777543,1366 days 02:39:28.077083,1.98,434
1,Rodent Abatement,532 days 21:57:25.640108,1365 days 20:09:28.077083,2.56,714
2,Health Condition,502 days 16:51:23.511865,2352 days 00:57:28.077083,4.67,92
3,Unsanitary Conditions,489 days 04:12:27.312751,1360 days 00:02:28.077083,2.78,157
4,Mold,468 days 00:10:14.274802,1365 days 23:21:28.077083,2.91,526
5,Farm Animals,463 days 18:37:11.713447,1238 days 22:37:28.077083,2.67,11
6,Insect/Vermin Infestation,442 days 02:29:52.934226,1364 days 01:50:28.077083,3.08,420
7,High Grass/Weeds,419 days 06:32:56.841789,1112 days 19:22:28.077083,2.65,340
8,Sewage Back Up,403 days 14:48:51.886607,1366 days 01:20:28.077083,3.38,63
9,Other,361 days 12:22:08.698201,1351 days 21:56:28.077083,3.73,161


**Findings:** Of the complaints currently open, the complaint type with the longest average open time is "food complaints" with an average duration of 688 days. Having a max-age-to-average-age ratio of 1.98 denotes fairly consistent data from an above average sample size (434). 

"Health condition" can be seen having the longest open complaint by far at 2351 days, 985 days longer than the second place complaint type. But with a ratio of 4.67 and a below average sample size of 92, this shows that the longest open "health condition" complaint is more of an outlier, possibly needing further investigation in the future.

**Note:** Complaint types with a smaller sample size are more prone to having a higher ratio of longest open complaint to average complaint age, from fewer outlier data points. A ratio closer to 1 denotes less "noisy" and more consistent data points throughout the sample size.

## Summary
This analysis of the Cleveland Department of Health complaints data set out to find not only the most common complaints submitted to the city, but also the open complaints that have the highest average time open. The findings here show that excessive accumulations of garbage are by far the most common complaint submitted, and of the currently open complaints, "food complaints" remain open the longest. Given this information, steps can be now be taken to either remedy or further investigate these two pain points. Once action has been taken, this same analysis can be redone to continue improving the public health of the city of Cleveland.